# 04: XGBoostモデルの学習

学習データを使用して、6つの統合モデルを学習します。

## 目的
- XGBoostモデルの学習スクリプトの実装
- 6つの統合モデル（軸数字予測2モデル + 組み合わせ予測4モデル）の学習
- モデル評価の実施
- 学習済みモデルの保存

## モデル構成
- **軸数字予測モデル（2モデル）**:
  - N3軸数字予測モデル（n3_axis.pkl）
  - N4軸数字予測モデル（n4_axis.pkl）
- **組み合わせ予測モデル（4モデル）**:
  - N3ボックス組み合わせ予測モデル（n3_box_comb.pkl）
  - N3ストレート組み合わせ予測モデル（n3_straight_comb.pkl）
  - N4ボックス組み合わせ予測モデル（n4_box_comb.pkl）
  - N4ストレート組み合わせ予測モデル（n4_straight_comb.pkl）


In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# プロジェクトルートのパスを設定
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = DATA_DIR / 'models'

print(f"プロジェクトルート: {PROJECT_ROOT}")
print(f"データディレクトリ: {DATA_DIR}")
print(f"モデルディレクトリ: {MODELS_DIR}")


In [ ]:
# XGBoostハイパーパラメータ設定
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'max_depth': 6,
    'learning_rate': 0.05,
    'n_estimators': 500,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'n_jobs': -1
}

print("XGBoostハイパーパラメータ:")
for key, value in xgb_params.items():
    print(f"  {key}: {value}")


## モデル評価関数の定義

AUC-ROC、Precision、Recall、F1-Score、Top-K Accuracyを計算します。


In [ ]:
def calculate_top_k_accuracy(y_true, y_pred_proba, k=5):
    """Top-K Accuracyを計算する
    
    Args:
        y_true: 真のラベル（形状: (n_samples,)）
        y_pred_proba: 予測確率（形状: (n_samples,)）
        k: 上位K件
    
    Returns:
        Top-K Accuracy（0-1）
    """
    # 予測確率が高い順にインデックスをソート
    sorted_indices = np.argsort(y_pred_proba)[::-1]
    
    # 上位K件のインデックス
    top_k_indices = sorted_indices[:k]
    
    # 上位K件中に正解（ラベル=1）が含まれているか
    return 1.0 if np.any(y_true[top_k_indices] == 1) else 0.0


def evaluate_model(model, X_val, y_val, model_name=""):
    """モデルを評価する
    
    Args:
        model: 学習済みモデル
        X_val: 検証データの特徴量
        y_val: 検証データのラベル
        model_name: モデル名（ログ出力用）
    
    Returns:
        評価結果の辞書
    """
    # 予測
    y_pred = model.predict(X_val)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    # 評価指標を計算
    auc_roc = roc_auc_score(y_val, y_pred_proba)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    
    # Top-K Accuracyを計算（K=5）
    top_k_acc = calculate_top_k_accuracy(y_val, y_pred_proba, k=5)
    
    results = {
        'model_name': model_name,
        'auc_roc': auc_roc,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'top_5_accuracy': top_k_acc
    }
    
    print(f"\n=== {model_name} の評価結果 ===")
    print(f"AUC-ROC: {auc_roc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Top-5 Accuracy: {top_k_acc:.4f}")
    
    return results


In [ ]:
# 軸数字予測モデルの学習
axis_models = {}
axis_evaluation_results = []

for target in ['n3', 'n4']:
    data_file = MODELS_DIR / f'{target}_axis_data.pkl'
    
    if not data_file.exists():
        print(f"警告: データファイルが見つかりません: {data_file}")
        continue
    
    print(f"\n{'='*60}")
    print(f"N{target[1].upper()}軸数字予測モデルの学習を開始します")
    print(f"{'='*60}")
    
    # データを読み込む
    with open(data_file, 'rb') as f:
        data = pickle.load(f)
    
    X_train = data['X_train']
    X_val = data['X_val']
    y_train = data['y_train']
    y_val = data['y_val']
    
    print(f"\nデータ形状:")
    print(f"  学習データ: {X_train.shape}")
    print(f"  検証データ: {X_val.shape}")
    print(f"  ラベル分布（学習）: {np.bincount(y_train)}")
    print(f"  ラベル分布（検証）: {np.bincount(y_val)}")
    
    # モデルを学習させる
    model = xgb.XGBClassifier(**xgb_params)
    
    print(f"\nモデルを学習中...")
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=True
    )
    
    # モデルを評価する
    model_name = f"{target}_axis"
    results = evaluate_model(model, X_val, y_val, model_name)
    axis_evaluation_results.append(results)
    
    # モデルを保存
    axis_models[model_name] = model
    
    print(f"\nモデル学習完了: {model_name}")

print(f"\n{'='*60}")
print(f"軸数字予測モデルの学習が完了しました")
print(f"{'='*60}")


In [ ]:
# 組み合わせ予測モデルの学習
comb_models = {}
comb_evaluation_results = []

for target in ['n3', 'n4']:
    for combo_type in ['box', 'straight']:
        data_file = MODELS_DIR / f'{target}_{combo_type}_comb_data.pkl'
        
        if not data_file.exists():
            print(f"警告: データファイルが見つかりません: {data_file}")
            continue
        
        print(f"\n{'='*60}")
        print(f"N{target[1].upper()} {combo_type.upper()}組み合わせ予測モデルの学習を開始します")
        print(f"{'='*60}")
        
        # データを読み込む
        with open(data_file, 'rb') as f:
            data = pickle.load(f)
        
        X_train = data['X_train']
        X_val = data['X_val']
        y_train = data['y_train']
        y_val = data['y_val']
        
        print(f"\nデータ形状:")
        print(f"  学習データ: {X_train.shape}")
        print(f"  検証データ: {X_val.shape}")
        print(f"  ラベル分布（学習）: {np.bincount(y_train)}")
        print(f"  ラベル分布（検証）: {np.bincount(y_val)}")
        
        # ラベルが全て同じ場合はスキップ（学習不可能）
        if len(np.unique(y_train)) == 1:
            print(f"\n⚠️ 警告: ラベルが全て同じです（全て{np.unique(y_train)[0]}）。このモデルはスキップします。")
            continue
        
        # モデルを学習させる
        model = xgb.XGBClassifier(**xgb_params)
        
        print(f"\nモデルを学習中...")
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=True
        )
        
        # モデルを評価する
        model_name = f"{target}_{combo_type}_comb"
        results = evaluate_model(model, X_val, y_val, model_name)
        comb_evaluation_results.append(results)
        
        # モデルを保存
        comb_models[model_name] = model
        
        print(f"\nモデル学習完了: {model_name}")

print(f"\n{'='*60}")
print(f"組み合わせ予測モデルの学習が完了しました")
print(f"{'='*60}")


## 評価結果のサマリー

すべてのモデルの評価結果をまとめます。


In [ ]:
# 評価結果のサマリー
print("\n" + "="*60)
print("全モデルの評価結果サマリー")
print("="*60)

all_results = axis_evaluation_results + comb_evaluation_results

results_df = pd.DataFrame(all_results)
print("\n評価指標の一覧:")
print(results_df.to_string(index=False))

print("\n各評価指標の平均値:")
print(f"  AUC-ROC: {results_df['auc_roc'].mean():.4f}")
print(f"  Precision: {results_df['precision'].mean():.4f}")
print(f"  Recall: {results_df['recall'].mean():.4f}")
print(f"  F1-Score: {results_df['f1_score'].mean():.4f}")
print(f"  Top-5 Accuracy: {results_df['top_5_accuracy'].mean():.4f}")

# 目標値との比較
print("\n目標値との比較:")
print(f"  AUC-ROC目標: 0.65以上")
for _, row in results_df.iterrows():
    status = "✅" if row['auc_roc'] >= 0.65 else "❌"
    print(f"    {row['model_name']}: {row['auc_roc']:.4f} {status}")


## 学習済みモデルの保存

6つのモデルを.pkl形式で保存します。


In [ ]:
# モデル保存ディレクトリの確認
if not MODELS_DIR.exists():
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 軸数字予測モデルの保存
print("\n軸数字予測モデルを保存中...")
for model_name, model in axis_models.items():
    model_file = MODELS_DIR / f"{model_name}.pkl"
    with open(model_file, 'wb') as f:
        pickle.dump(model, f)
    print(f"  保存完了: {model_file}")

# 組み合わせ予測モデルの保存
print("\n組み合わせ予測モデルを保存中...")
for model_name, model in comb_models.items():
    model_file = MODELS_DIR / f"{model_name}.pkl"
    with open(model_file, 'wb') as f:
        pickle.dump(model, f)
    print(f"  保存完了: {model_file}")

# 評価結果も保存
eval_results_file = MODELS_DIR / 'evaluation_results.pkl'
with open(eval_results_file, 'wb') as f:
    pickle.dump({
        'axis_results': axis_evaluation_results,
        'comb_results': comb_evaluation_results,
        'all_results': all_results
    }, f)
print(f"\n評価結果を保存しました: {eval_results_file}")

print(f"\nすべてのモデルを保存しました: {MODELS_DIR}")


## 次のステップ

モデル学習が完了しました。次のステップとして、モデル読み込みユーティリティを作成し、推論機能を実装します。
